# Week 4: LLM Benchmarking — Water Supply Volume Forecasting

**Team:** AI Delinquents  
**Task:** Benchmark locally hosted LLMs on the same regression problem as Week 3 — predicting April–September naturalized streamflow volume at The Dalles, OR (kcfs-days).

**Structure:**
1. Setup and data loading
2. Prompt design and few-shot example selection
3. Prompt iteration (v1 zero-shot vs v2 few-shot) on a small subset
4. Full benchmark across all three model endpoints
5. Metrics: NSE, KGE, RMSE vs Week 3 ML models
6. Error analysis and failure inspection
7. Save outputs and draft report text

**Before running:** Set `DRY_RUN = False` in Cell 2 to use real LLM endpoints (requires JupyterHub with localhost:8000-8002 active).

## Step 1: Imports

In [ ]:
import json
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

openai = __import__('openai')
OpenAI = openai.OpenAI

## Step 2: Paths, Settings, and DRY_RUN Flag

Set `DRY_RUN = True` to run the notebook without live LLM endpoints (returns deterministic mock predictions). Set `DRY_RUN = False` on JupyterHub to use real models.

In [ ]:
# ── Toggle this before running on JupyterHub ────────────────────────────────
DRY_RUN = True   # False = use real endpoints; True = mock responses for testing
# ────────────────────────────────────────────────────────────────────────────

# Paths (relative to this notebook at week_4_files/Notebooks/)
ROOT          = Path('..').resolve()                        # week_4_files/
WEEK3_OUTPUTS = ROOT.parent.parent / 'ai_delinquents' / 'week_3_files' / 'outputs'
OUTPUT_DIR    = ROOT / 'Results'
PROMPTS_DIR   = ROOT / 'Prompts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

# Verify week 3 artifacts are accessible
for p in [WEEK3_OUTPUTS / 'test_split.csv',
          WEEK3_OUTPUTS / 'test_predictions.csv',
          WEEK3_OUTPUTS / 'train_split.csv']:
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'  {status}  {p.name}')

# Model endpoints (locally hosted on JupyterHub)
MODEL_ENDPOINTS = [
    {'label': 'phi-3.5-mini-instruct',  'model_name': 'Phi-3.5-mini-instruct',  'host': 'localhost', 'port': 8000},
    {'label': 'phi-mini-moe-instruct',  'model_name': 'Phi-mini-MoE-instruct',  'host': 'localhost', 'port': 8001},
    {'label': 'gemma-3-12b-it',         'model_name': 'gemma-3-12b-it',         'host': 'localhost', 'port': 8002},
]
API_KEY          = 'EMPTY'
SYSTEM_PROMPT    = 'You are a careful hydrologist. Follow output format instructions exactly.'
ITERATION_NUMBER = 1
RANDOM_SEED      = 42

print(f'\nDRY_RUN = {DRY_RUN}')
print(f'Results will be saved to: {OUTPUT_DIR}')

## Step 3: Load Week 3 Artifacts

We use the **same held-out test split** (WY 2013–2018) and the **same ML predictions** from Week 3 to ensure a fair, apples-to-apples comparison.

In [ ]:
test_df  = pd.read_csv(WEEK3_OUTPUTS / 'test_split.csv')
train_df = pd.read_csv(WEEK3_OUTPUTS / 'train_split.csv')
ml_preds = pd.read_csv(WEEK3_OUTPUTS / 'test_predictions.csv')

FEATURE_COLS = [
    'apr1_swe_inches', 'apr1_swe_anomaly_pct',
    'djf_pdo', 'djf_nino34', 'djf_pna',
    'jan_mar_mean_q_cfs', 'oct_mar_volume_kcfs_days',
]
TARGET_COL = 'target_volume'

print(f'Test years:  {test_df["water_year"].tolist()}')
print(f'Train years: {len(train_df)} ({train_df["water_year"].min()}–{train_df["water_year"].max()})')
print(f'\nTest split preview:')
display(test_df[['water_year', TARGET_COL] + FEATURE_COLS])

## Step 4: Few-Shot Example Selection

Select 3 representative training years (dry / median / wet) to use as few-shot examples. Examples are selected programmatically from the training set terciles — **not** from the test set — to prevent leakage.

In [ ]:
def select_few_shot_years(train_df, n=3, seed=RANDOM_SEED):
    """Select dry, median, and wet representative training years."""
    df = train_df.sort_values(TARGET_COL).reset_index(drop=True)
    n_train = len(df)
    # Tercile representatives: ~17th, 50th, ~83rd percentile
    indices = [
        int(n_train * 0.17),
        int(n_train * 0.50),
        int(n_train * 0.83),
    ]
    return df.iloc[indices].reset_index(drop=True)

few_shot_df = select_few_shot_years(train_df)
print('Few-shot examples selected:')
display(few_shot_df[['water_year', TARGET_COL] + FEATURE_COLS])
print(f'\nTarget range in training: {train_df[TARGET_COL].min():,.0f} – {train_df[TARGET_COL].max():,.0f} kcfs-days')
print(f'Training median: {train_df[TARGET_COL].median():,.0f} kcfs-days')

## Step 5: Prompt Design and Parser

### Design principles
- **Explicit output format**: `{"prediction_kcfs_days": <number>}` — no free text
- **No leakage**: true test values never appear in prompts or few-shot examples
- **Domain grounding**: features described in plain English with units and anomaly context
- **Two versions**: v1 (zero-shot) and v2 (few-shot with 3 training years)

In [ ]:
def format_features(row):
    """Convert a feature row into a human-readable narrative string."""
    enso_desc = (
        'El Niño (warm)' if row['djf_nino34'] > 0.5
        else 'La Niña (cool)' if row['djf_nino34'] < -0.5
        else 'ENSO-neutral'
    )
    return (
        f"- April 1 basin-average SWE: {row['apr1_swe_inches']:.1f} inches "
        f"({row['apr1_swe_anomaly_pct']:+.1f}% vs median)\n"
        f"- DJF Nino3.4: {row['djf_nino34']:.2f} ({enso_desc})\n"
        f"- DJF PDO: {row['djf_pdo']:.2f}\n"
        f"- DJF PNA: {row['djf_pna']:.2f}\n"
        f"- Jan–Mar mean naturalized flow: {row['jan_mar_mean_q_cfs']:,.0f} cfs\n"
        f"- Oct–Mar cumulative volume: {row['oct_mar_volume_kcfs_days']:,.0f} kcfs-days"
    )


def build_prompt(row, version='v1', few_shot_df=None):
    """
    Build a prompt for a single test year.
    version='v1': zero-shot
    version='v2_few_shot': includes 3 training examples
    """
    header = (
        'You are a hydrologist forecasting seasonal water supply for the Columbia River Basin.\n'
        'Given the climate and snowpack conditions listed below for a water year, '
        'predict the total April–September naturalized streamflow volume at The Dalles, OR.\n\n'
        'Rules:\n'
        '- Return ONLY valid JSON: {"prediction_kcfs_days": <positive number>}\n'
        '- prediction_kcfs_days must be a positive number in kcfs-days\n'
        '- No explanation, no extra text, no markdown\n'
        f'- Typical range for this location: 28,000–71,000 kcfs-days\n'
    )

    if version == 'v2_few_shot' and few_shot_df is not None:
        examples = ['\nExamples from historical training years (do NOT use these years in your prediction):']
        for _, ex in few_shot_df.iterrows():
            examples.append(
                f'\nWater year {int(ex["water_year"])} conditions:\n'
                + format_features(ex)
                + f'\nActual Apr–Sep volume: {{"prediction_kcfs_days": {ex[TARGET_COL]:,.0f}}}'
            )
        header += '\n'.join(examples)

    input_block = (
        f'\n\nPredict this water year (do NOT reveal the true value):\n'
        f'Water year {int(row["water_year"])} conditions:\n'
        + format_features(row)
        + '\n\nReturn JSON only:'
    )
    return header + input_block


def parse_response(raw_text):
    """Extract prediction_kcfs_days from LLM response JSON. Returns dict."""
    out = {'prediction_kcfs_days': np.nan, 'parse_ok': False, 'parse_error': None}

    if not isinstance(raw_text, str) or not raw_text.strip():
        out['parse_error'] = 'empty_response'
        return out

    # Extract JSON object from response (handles extra whitespace/markdown fences)
    match = re.search(r'\{[^{}]*\}', raw_text.strip(), flags=re.DOTALL)
    candidate = match.group(0) if match else raw_text.strip()

    try:
        payload = json.loads(candidate)
    except Exception:
        out['parse_error'] = 'invalid_json'
        return out

    raw_val = payload.get('prediction_kcfs_days')
    if raw_val is None:
        out['parse_error'] = 'missing_key'
        return out

    try:
        val = float(raw_val)
    except Exception:
        out['parse_error'] = 'non_numeric'
        return out

    if val <= 0 or val > 500_000:
        out['parse_error'] = 'out_of_range'
        return out

    out['prediction_kcfs_days'] = val
    out['parse_ok'] = True
    return out


# Print example prompt for inspection
example_row = test_df.iloc[0]
print('--- ZERO-SHOT PROMPT (v1) ---')
print(build_prompt(example_row, version='v1'))
print('\n--- FEW-SHOT PROMPT (v2) ---')
print(build_prompt(example_row, version='v2_few_shot', few_shot_df=few_shot_df))

## Step 6: LLM Query Function (with DRY_RUN mode)

In [ ]:
def _mock_response(row, seed=0):
    """Deterministic fake LLM response for DRY_RUN testing."""
    rng = random.Random(seed + int(row['water_year']) * 7)
    # Mock prediction: true value ± 10-30% random noise
    noise = rng.uniform(-0.25, 0.25)
    fake = row[TARGET_COL] * (1 + noise)
    return json.dumps({'prediction_kcfs_days': round(fake, 1)})


def query_llm(prompt, endpoint_cfg, row=None, temperature=0.0, seed=0):
    """Query LLM endpoint. Returns raw response string."""
    if DRY_RUN:
        return _mock_response(row, seed=seed)

    client = OpenAI(
        api_key=API_KEY,
        base_url=f"http://{endpoint_cfg['host']}:{endpoint_cfg['port']}/v1"
    )
    response = client.chat.completions.create(
        model=endpoint_cfg['model_name'],
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ],
        temperature=temperature,
        seed=seed,
    )
    return response.choices[0].message.content


print('query_llm defined.' + ('  [DRY_RUN mode — no real endpoints called]' if DRY_RUN else '  [LIVE mode]'))

## Step 7: Connectivity Smoke Test

Test one example against each endpoint before running the full benchmark.

In [ ]:
smoke_row = test_df.iloc[0]
smoke_prompt = build_prompt(smoke_row, version='v1')

for endpoint_cfg in MODEL_ENDPOINTS:
    raw = query_llm(smoke_prompt, endpoint_cfg, row=smoke_row, temperature=0.0, seed=0)
    parsed = parse_response(raw)
    print(f"\nModel: {endpoint_cfg['model_name']} (port {endpoint_cfg['port']})")
    print(f'  Raw:    {raw.strip()}')
    print(f'  Parsed: {parsed}')
    print(f'  True:   {smoke_row[TARGET_COL]:,.0f} kcfs-days')

## Step 8: Prompt Iteration — Compare v1 (Zero-Shot) vs v2 (Few-Shot)

Run both prompt versions on a 4-year subset of the test set to evaluate parse reliability and directional accuracy before the full benchmark. This informs which prompt version to use in the final run.

In [ ]:
# Use all 6 test years for iteration (small enough to run quickly)
subset = test_df.copy()
PROMPT_VERSIONS = ['v1', 'v2_few_shot']

iteration_records = []
for endpoint_cfg in MODEL_ENDPOINTS:
    for version in PROMPT_VERSIONS:
        for _, row in subset.iterrows():
            prompt = build_prompt(
                row, version=version,
                few_shot_df=few_shot_df if version == 'v2_few_shot' else None
            )
            raw = query_llm(prompt, endpoint_cfg, row=row, temperature=0.0, seed=RANDOM_SEED)
            parsed = parse_response(raw)
            iteration_records.append({
                'water_year':    int(row['water_year']),
                'model_label':   endpoint_cfg['label'],
                'version':       version,
                'true_volume':   row[TARGET_COL],
                'raw_response':  raw,
                **parsed,
            })

iter_df = pd.DataFrame(iteration_records)
iter_df['abs_error'] = (iter_df['prediction_kcfs_days'] - iter_df['true_volume']).abs()

iter_summary = (
    iter_df.groupby(['model_label', 'version'])
    .agg(
        parse_success_rate=('parse_ok', 'mean'),
        mean_abs_error=('abs_error', 'mean'),
        rows=('parse_ok', 'size')
    )
    .reset_index()
    .sort_values(['model_label', 'parse_success_rate', 'mean_abs_error'],
                 ascending=[True, False, True])
)

print('Prompt iteration summary (parse success + mean absolute error):')
display(iter_summary)

In [ ]:
# Identify best-performing version per model
best_version_by_model = (
    iter_summary
    .sort_values(['model_label', 'parse_success_rate', 'mean_abs_error'],
                 ascending=[True, False, True])
    .groupby('model_label', as_index=False)
    .first()[['model_label', 'version']]
)
print('Best prompt version per model:')
display(best_version_by_model)

# Inspect any parse failures
failures = iter_df[~iter_df['parse_ok']]
print(f'\nParse failures: {len(failures)} / {len(iter_df)}')
if not failures.empty:
    display(failures[['model_label', 'version', 'water_year', 'raw_response', 'parse_error']])

## Step 9: Full Benchmark Run

Run the best-performing prompt version (from Step 8) for each model across all 6 test years. Also run the full run with the best overall version for the combined results CSV.

In [ ]:
# Merge best version choice into endpoint config
best_version_map = dict(zip(best_version_by_model['model_label'], best_version_by_model['version']))

benchmark_records = []
for endpoint_cfg in MODEL_ENDPOINTS:
    best_v = best_version_map.get(endpoint_cfg['label'], 'v1')
    print(f"Running {endpoint_cfg['model_name']}  (prompt: {best_v})...")
    for _, row in test_df.iterrows():
        prompt = build_prompt(
            row, version=best_v,
            few_shot_df=few_shot_df if best_v == 'v2_few_shot' else None
        )
        raw = query_llm(prompt, endpoint_cfg, row=row, temperature=0.0, seed=RANDOM_SEED)
        parsed = parse_response(raw)

        # Merge in ML predictions for this year
        ml_row = ml_preds[ml_preds['water_year'] == int(row['water_year'])].iloc[0]

        benchmark_records.append({
            'water_year':              int(row['water_year']),
            'model_label':             endpoint_cfg['label'],
            'model_name':              endpoint_cfg['model_name'],
            'endpoint_port':           endpoint_cfg['port'],
            'prompt_version':          best_v,
            'observed':                row[TARGET_COL],
            'pred_climatology':        ml_row['pred_climatology'],
            'pred_mlr':                ml_row['pred_mlr'],
            'pred_xgb':                ml_row['pred_xgb'],
            'raw_response':            raw,
            **parsed,
        })

results_df = pd.DataFrame(benchmark_records)
print(f'\nBenchmark complete: {len(results_df)} rows ({len(test_df)} years × {len(MODEL_ENDPOINTS)} models)')
display(results_df[['water_year', 'model_label', 'observed', 'pred_mlr', 'pred_xgb',
                     'prediction_kcfs_days', 'parse_ok']].head(12))

## Step 10: Metrics — NSE, KGE, RMSE

Compute the same metrics as Week 3 for all ML models and each LLM, on the same 6 held-out test years.

In [ ]:
def nse(obs, pred):
    obs, pred = np.array(obs), np.array(pred)
    return float(1 - np.sum((obs - pred)**2) / np.sum((obs - np.mean(obs))**2))

def kge(obs, pred):
    obs, pred = np.array(obs), np.array(pred)
    r = np.corrcoef(obs, pred)[0, 1]
    alpha = np.std(pred) / np.std(obs)
    beta  = np.mean(pred) / np.mean(obs)
    return float(1 - np.sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2))

def rmse(obs, pred):
    return float(np.sqrt(np.mean((np.array(obs) - np.array(pred))**2)))

def mae(obs, pred):
    return float(np.mean(np.abs(np.array(obs) - np.array(pred))))

def skill_score(obs, pred, baseline):
    mse_m = np.mean((np.array(obs) - np.array(pred))**2)
    mse_b = np.mean((np.array(obs) - np.array(baseline))**2)
    return float(1 - mse_m / mse_b)


# Compute metrics for ML models (using first model's rows for observed + ML preds)
ref = results_df[results_df['model_label'] == results_df['model_label'].iloc[0]].copy()
obs = ref['observed'].values
clim = ref['pred_climatology'].values

metric_rows = []
for m_name, pred_col in [('Climatology (Week 3)', 'pred_climatology'),
                          ('MLR (Week 3)',          'pred_mlr'),
                          ('XGBoost (Week 3)',       'pred_xgb')]:
    pred = ref[pred_col].values
    metric_rows.append({
        'model': m_name, 'model_type': 'ML',
        'nse': nse(obs, pred), 'kge': kge(obs, pred),
        'rmse': rmse(obs, pred), 'mae': mae(obs, pred),
        'skill_vs_clim': skill_score(obs, pred, clim) if 'Clim' not in m_name else np.nan,
        'parse_success_rate': 1.0, 'n_years': len(obs),
    })

# Compute metrics for each LLM
for model_label, grp in results_df.groupby('model_label'):
    valid = grp[grp['parse_ok']].copy()
    if len(valid) < 2:
        print(f'  WARNING: {model_label} has fewer than 2 valid predictions — skipping metrics')
        continue
    obs_v   = valid['observed'].values
    pred_v  = valid['prediction_kcfs_days'].values
    clim_v  = valid['pred_climatology'].values
    metric_rows.append({
        'model': grp['model_name'].iloc[0],
        'model_type': 'LLM',
        'nse':  nse(obs_v, pred_v),
        'kge':  kge(obs_v, pred_v),
        'rmse': rmse(obs_v, pred_v),
        'mae':  mae(obs_v, pred_v),
        'skill_vs_clim': skill_score(obs_v, pred_v, clim_v),
        'parse_success_rate': grp['parse_ok'].mean(),
        'n_years': len(valid),
    })

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(3))

## Step 11: Visualization — Observed vs Predicted

Side-by-side comparison of Week 3 ML models and all LLMs on the 6 test years.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

years = test_df['water_year'].values
obs_vals = test_df[TARGET_COL].values

# ── Left: Time series ────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(years, obs_vals, 'ko-', lw=2, ms=7, label='Observed', zorder=5)
ax.plot(years, ref['pred_mlr'].values,  's--', color='#ff7f0e', lw=1.5, ms=6, label='MLR (Week 3)')
ax.plot(years, ref['pred_xgb'].values,  '^--', color='#1f77b4', lw=1.5, ms=6, label='XGBoost (Week 3)')
colors_llm = ['#2ca02c', '#9467bd', '#8c564b']
for (model_label, grp), c in zip(results_df.groupby('model_label'), colors_llm):
    valid = grp[grp['parse_ok']]
    label_str = grp['model_name'].iloc[0].replace('instruct','').strip()
    ax.plot(valid['water_year'], valid['prediction_kcfs_days'],
            'D:', color=c, lw=1.2, ms=5, label=f'LLM: {label_str}')
ax.set_xlabel('Water Year')
ax.set_ylabel('Apr–Sep Volume (kcfs-days)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.set_title('Observed vs Predicted — Test Set (WY 2013–2018)')
ax.legend(fontsize=8, loc='upper left')

# ── Right: 1:1 scatter ───────────────────────────────────────────────────────
ax = axes[1]
lims = [min(obs_vals) * 0.9, max(obs_vals) * 1.1]
ax.plot(lims, lims, 'k--', lw=1, label='1:1')
ax.scatter(obs_vals, ref['pred_mlr'].values,  s=70, marker='s', color='#ff7f0e',
           edgecolors='white', label='MLR', zorder=4)
ax.scatter(obs_vals, ref['pred_xgb'].values,  s=70, marker='^', color='#1f77b4',
           edgecolors='white', label='XGBoost', zorder=4)
for (model_label, grp), c in zip(results_df.groupby('model_label'), colors_llm):
    valid = grp[grp['parse_ok']]
    label_str = grp['model_name'].iloc[0].replace('instruct','').strip()
    ax.scatter(valid['observed'], valid['prediction_kcfs_days'],
               s=60, marker='D', color=c, edgecolors='white',
               label=f'LLM: {label_str}', zorder=3)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Observed (kcfs-days)')
ax.set_ylabel('Predicted (kcfs-days)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.set_title('1:1 Scatter — ML vs LLM')
ax.legend(fontsize=8)

fig.tight_layout()
fig_path = OUTPUT_DIR / f'llm_vs_ml_comparison_{ITERATION_NUMBER}.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {fig_path}')

## Step 12: Error Analysis and Failure Inspection

In [ ]:
# Per-year residuals for each model
print('Residuals by year (predicted − observed, kcfs-days):')
residuals = pd.DataFrame({'water_year': years, 'observed': obs_vals})
residuals['mlr_error']  = ref['pred_mlr'].values  - obs_vals
residuals['xgb_error']  = ref['pred_xgb'].values  - obs_vals
for model_label, grp in results_df.groupby('model_label'):
    valid = grp[grp['parse_ok']].set_index('water_year')
    residuals[f"{model_label}_error"] = [
        valid.loc[y, 'prediction_kcfs_days'] - obs_vals[i]
        if y in valid.index else np.nan
        for i, y in enumerate(years)
    ]
display(residuals.round(0))

# Parse failures
failures = results_df[~results_df['parse_ok']]
print(f'\nParse failures: {len(failures)} / {len(results_df)}')
if not failures.empty:
    display(failures[['model_label', 'water_year', 'raw_response', 'parse_error']])

# Largest errors per LLM
print('\nLargest absolute errors (LLMs only):')
llm_valid = results_df[results_df['parse_ok']].copy()
llm_valid['abs_error'] = (llm_valid['prediction_kcfs_days'] - llm_valid['observed']).abs()
display(
    llm_valid.nlargest(5, 'abs_error')[
        ['model_label', 'water_year', 'observed', 'prediction_kcfs_days', 'abs_error']
    ].round(0)
)

## Step 13: Save Results and Prompt Artifacts

In [ ]:
# Save per-model clean CSVs (competition format)
clean_cols = [
    'water_year', 'model_label', 'model_name', 'endpoint_port', 'prompt_version',
    'observed', 'pred_mlr', 'pred_xgb', 'prediction_kcfs_days', 'parse_ok', 'parse_error'
]
for model_label, grp in results_df.groupby('model_label'):
    clean_path = OUTPUT_DIR / f'{model_label}_results_clean_{ITERATION_NUMBER}.csv'
    raw_path   = OUTPUT_DIR / f'{model_label}_results_raw_{ITERATION_NUMBER}.csv'
    grp[clean_cols].to_csv(clean_path, index=False)
    grp.to_csv(raw_path, index=False)
    print(f'Saved: {clean_path.name}')
    print(f'Saved: {raw_path.name}')

# Save combined clean CSV
combined_path = OUTPUT_DIR / f'llm_results_clean_{ITERATION_NUMBER}.csv'
results_df[clean_cols].to_csv(combined_path, index=False)
print(f'\nSaved combined: {combined_path.name}')

# Save metrics CSV
metrics_path = OUTPUT_DIR / f'metrics_summary_{ITERATION_NUMBER}.csv'
metrics_df.to_csv(metrics_path, index=False)
print(f'Saved metrics:  {metrics_path.name}')

In [ ]:
# Save prompt artifacts to Prompts/
overall_best_v = (
    iter_summary
    .sort_values(['parse_success_rate', 'mean_abs_error'], ascending=[False, True])
    .iloc[0]['version']
)

# prompt_template.txt — canonical template for the overall best version
template_text = build_prompt(
    test_df.iloc[0].copy().rename({'water_year': 'water_year', TARGET_COL: TARGET_COL}),
    version=overall_best_v,
    few_shot_df=few_shot_df if overall_best_v == 'v2_few_shot' else None
)
template_text = template_text.replace(
    f"Water year {int(test_df.iloc[0]['water_year'])} conditions:",
    'Water year [YEAR] conditions:'
)
(PROMPTS_DIR / 'prompt_template.txt').write_text(template_text, encoding='utf-8')
print(f'Saved: Prompts/prompt_template.txt  (version: {overall_best_v})')

# few_shot_samples.json
few_shot_list = [
    {
        'water_year': int(row['water_year']),
        'features':   {c: round(row[c], 4) for c in FEATURE_COLS},
        'target_volume_kcfs_days': round(row[TARGET_COL], 1),
        'narrative':  format_features(row)
    }
    for _, row in few_shot_df.iterrows()
]
(PROMPTS_DIR / 'few_shot_samples.json').write_text(
    json.dumps(few_shot_list, indent=2), encoding='utf-8'
)
print('Saved: Prompts/few_shot_samples.json')

## Step 14: Draft Report Text for README

In [ ]:
best_llm = metrics_df[metrics_df['model_type'] == 'LLM'].sort_values('rmse').iloc[0]
mlr_row  = metrics_df[metrics_df['model'] == 'MLR (Week 3)'].iloc[0]
xgb_row  = metrics_df[metrics_df['model'] == 'XGBoost (Week 3)'].iloc[0]

approach_text = f"""
## Approach

We benchmarked three locally hosted LLMs (Phi-3.5-mini-instruct, Phi-mini-MoE-instruct, gemma-3-12b-it)
on the same regression task as Week 3: predicting April–September naturalized flow volume at The Dalles, OR
(target unit: kcfs-days). Each LLM received a structured natural-language prompt describing the seven
predictor variables for a given water year and was asked to return a JSON object with a single numeric
prediction. We tested two prompt versions: zero-shot (v1) and 3-example few-shot (v2), selecting
few-shot examples from the training set (WY 1985–2012) by tercile rank to represent dry, median, and wet
years. All evaluation used the same held-out test split as Week 3 (WY 2013–2018, n=6).

### Repeatability
1. `pixi install` (or `pip install -r requirements.txt`)
2. Confirm LLM endpoints are running at localhost:8000-8002
3. Set `DRY_RUN = False` in Cell 2 of `Notebooks/llm_benchmark.ipynb`
4. Run all cells in order — outputs are saved to `Results/`
"""

error_text = f"""
## Error Analysis

Best LLM RMSE: {best_llm['rmse']:,.0f} kcfs-days ({best_llm['model']}) vs
MLR RMSE: {mlr_row['rmse']:,.0f} kcfs-days and XGBoost RMSE: {xgb_row['rmse']:,.0f} kcfs-days.
LLMs showed [describe pattern — e.g., regression to the mean, over/under-prediction in extreme years].
Parse failure rate: [fill in from run]. Primary failure mode: [e.g., verbose JSON with extra keys,
units included in number string, markdown code fences wrapping JSON].
WY 2015 (record-low SWE, -51% anomaly) was the hardest year for all models;
LLMs may have anchored on the strong El Niño signal while underweighting the extreme SWE deficit.
"""

print(approach_text)
print(error_text)